In [1]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import numpy as np
import pandas as pd
import ast
import glob
import pickle
import dask
import os
import itertools


#from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score


from multiprocessing import Pool, cpu_count

import dask
import dask.dataframe as dd
from dask.distributed import Client
#client = Client(n_workers=20, memory_limit="10GB", interface='lo')
from concurrent.futures import ThreadPoolExecutor

import dask_ml.cluster as dask_cluster

from pprint import pprint
import os

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)


### Import Benchmark Files

In [3]:
benchmark_TLGRF_dataset = dd.read_csv("../generate_benchmark_TLGRF_dataset/benchmark_TLGRF_dataset.csv", assume_missing=True).compute()
benchmark_TLGRF_dataset["date"] = pd.to_datetime(benchmark_TLGRF_dataset["date"])
benchmark_TLGRF_dataset = benchmark_TLGRF_dataset[benchmark_TLGRF_dataset["log_rolled_cases"] >= np.log(21.1)]
benchmark_TLGRF_dataset = benchmark_TLGRF_dataset[benchmark_TLGRF_dataset["date"] <= "2022-12-31"]
benchmark_TLGRF_dataset = benchmark_TLGRF_dataset.sort_values(by=["fips","date"])
display(benchmark_TLGRF_dataset)

,fips,days_from_start,intercept_TLGRF,r_TLGRF,county,state,date,rolled_cases,log_rolled_cases,shifted_log_rolled_cases,TLGRF_predicted_log_rolled_cases
17,1001.0,86.0,NaN,NaN,Autauga,Alabama,2020-04-16,20.285714,3.062723,3.133629,NaN
18,1001.0,87.0,-26.766326,0.028066,Autauga,Alabama,2020-04-17,20.714286,3.082565,3.170286,3.279026
19,1001.0,88.0,-313.778047,0.007752,Autauga,Alabama,2020-04-18,20.714286,3.082565,3.205646,3.136830
20,1001.0,89.0,-60.990643,0.021119,Autauga,Alabama,2020-04-19,21.000000,3.095578,3.228543,3.243410
21,1001.0,90.0,-54.020959,0.022045,Autauga,Alabama,2020-04-20,21.428571,3.114784,3.256447,3.269100
...,...,...,...,...,...,...,...,...,...,...,...
10119,99999.0,1071.0,2837.736966,-0.006383,New York City,New York,2022-12-27,81539.428571,11.308855,11.249207,11.264175
10120,99999.0,1072.0,2628.429057,-0.007237,New York City,New York,2022-12-28,80849.571429,11.300359,11.241229,11.249699
10121,99999.0,1073.0,2338.714337,-0.008885,New York City,New York,2022-12-29,80031.285714,11.290187,11.234906,11.227992
10122,99999.0,1074.0,2540.040095,-0.007669,New York City,New York,2022-12-30,79315.142857,11.281198,11.229165,11.227516


In [4]:
individual_grf_df = dd.read_csv("individual_grf_df.csv", assume_missing=True, low_memory=False).compute()
individual_grf_df["date"] = pd.to_datetime(individual_grf_df["date"])
individual_grf_df = individual_grf_df[individual_grf_df["log_rolled_cases"] >= np.log(21.1)]
individual_grf_df = individual_grf_df[individual_grf_df["date"] <= "2022-12-31"]
individual_grf_df = individual_grf_df.sort_values(by=["fips","date"])
display(individual_grf_df)

,fips,days_from_start,r_individual_grf,state,county,date,log_rolled_cases,shifted_log_rolled_cases,individual_grf_predicted_log_rolled_cases
17,1001.0,86.0,NaN,Alabama,Autauga,2020-04-16,3.062723,3.133629,NaN
18,1001.0,87.0,0.002030,Alabama,Autauga,2020-04-17,3.082565,3.170286,3.096777
19,1001.0,88.0,-0.002771,Alabama,Autauga,2020-04-18,3.082565,3.205646,3.063169
20,1001.0,89.0,0.001940,Alabama,Autauga,2020-04-19,3.095578,3.228543,3.109158
21,1001.0,90.0,-0.005646,Alabama,Autauga,2020-04-20,3.114784,3.256447,3.075266
...,...,...,...,...,...,...,...,...,...
532762,99999.0,1071.0,NaN,New York,New York City,2022-12-27,11.308855,11.249207,NaN
532763,99999.0,1072.0,NaN,New York,New York City,2022-12-28,11.300359,11.241229,NaN
532764,99999.0,1073.0,NaN,New York,New York City,2022-12-29,11.290187,11.234906,NaN
532765,99999.0,1074.0,NaN,New York,New York City,2022-12-30,11.281198,11.229165,NaN
